---
title: "Internes Szenario- und Kalibrierungsdashboard"
execute:
  echo: true
format:
  html:
    code-fold: true
---

# Interner Arbeitsstand {#sec-internes-dashboard}

**Nur für die interne Redaktion. Kein Bestandteil der finalen Publikation.**
Das Profil `publikation` schließt diesen Anhang aus. Das Dashboard verändert weder
die wissenschaftlichen Referenznotebooks noch deren Ergebnisdateien oder QMD-Texte.

**Bedienung:** In der nächsten Zelle Excel-Pfad und nationales Szenario wählen,
Kernel neu starten und alle Zellen ausführen. Danach HTML neu rendern. Die HTML-Regler
rechnen unmittelbar mit diesem eingebetteten Datenstand; sie können keine lokale Excel
neu einlesen. Der JSON-Download sichert Parameter, Ergebnisse und Eingabe-Hashes.
Für eine Übernahme in den Bericht müssen die Referenznotebooks bewusst aktualisiert werden.

Quellen und Bilanzgrenzen: @sec-gebaeudebestand und @sec-allokation sowie die
wissenschaftlichen Anhänge. Verkehrsanteile beziehen sich auf **elektrische Energie**,
nicht auf Fahrten oder Personenkilometer. Alle Salden: Erzeugung minus Bedarf.


In [1]:
from pathlib import Path
import json, hashlib, sys
import numpy as np
import pandas as pd
from IPython.display import display, HTML
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'_quarto.yml').exists())
BASE = ROOT/'Berechnungen/gebaeudesektormodell'
sys.path.insert(0, str(BASE))
from utils import sha256

# Ausgangsszenario: Original-Pool (Strombed_estim + zuordnung_gfz)
# oder editierbare Reinschrift (Typen + GFZ_Zuordnung).
XLSX = ROOT/'Berechnungen/gebaeudesektormodell/data/Building-Sector_Pool_1.xlsx'
SCENARIO = 'transition_2040'
AREA_COLUMN = 'Nationale BGF 2040'
# Andere Flächenjahre sind kein zeitlich konsistenter Vergleich mit UBA 2040!
DEFAULT = dict(miv=.8, other_transport=0., free_pv=0., hydrogen=0.,
               saving=0., pv_scale=1., ngf_bgf=.8,
               density_f_res=30.4, density_f_ed=26.4, density_dx=.15, density_cutoff=62.)
# saving: proportionale Strombedarfssenkung des gewählten Gebäudepools.
# pv_scale: unabhängige Skalierung seiner gesamten PV-Erzeugung, keine PEExcel-Neusimulation.
# Freiflächen-PV bleibt gemäß bisheriger Allokationsregel separat unzugeordnet.


## Eingaben, Abdeckung und Herkunft

Die nationale BGF wird unverändert übernommen. Fehlende Flächen oder positive Flächen
ohne Energiekennwerte stoppen die Berechnung; eine explizite Nullfläche ist zulässig.
Excel-Formeln werden nicht neu berechnet: Verwendet werden die gespeicherten Werte.
Für neue Lieferungen müssen Bezeichnungen und Einheiten dem dokumentierten Datenvertrag
entsprechen: BGF in m², Kennwerte in kWh/(m² NGF·a). NGF/BGF = 0,8 ist vorläufig.


In [2]:
book = pd.ExcelFile(XLSX)
clean = 'Typen' in book.sheet_names
data = pd.read_excel(book, sheet_name='Typen' if clean else 'Strombed_estim').dropna(how='all')
mapping = pd.read_excel(book, sheet_name='GFZ_Zuordnung' if clean else 'zuordnung_gfz', usecols=range(6)).dropna(how='all')
book.close()
assert data.Archetype.notna().all() and data.Archetype.is_unique
data['Geometrie'] = data.Archetype.str.split('|', regex=False).str[0]
area = pd.to_numeric(data[AREA_COLUMN], errors='raise')
assert area.notna().all() and area.ge(0).all(), 'Fehlende/negative Fläche: Quelle ergänzen'
fields = ['Grid_to_building_min','PV_total_direct','PV_to_Egrid']
for col in fields:
    data[col] = pd.to_numeric(data[col], errors='raise')
    assert data.loc[area.gt(0),col].notna().all(), col
    assert data.loc[area.gt(0),col].ge(0).all(), col
data['BGF'] = area
data['demand'] = data.Grid_to_building_min + data.PV_total_direct
data['pv'] = data.PV_total_direct + data.PV_to_Egrid
national = {}
source_files = [XLSX]
for kind,filename in [('g','stromerzeugung_2040.csv'),('d','strombedarf_2040.csv')]:
    path=ROOT/'Daten/sektorenmodell'/filename
    table=pd.read_csv(path)
    selected=table.loc[table.scenario.eq(SCENARIO)]
    assert len(selected) and selected.metric.is_unique and selected.value_pj.notna().all()
    assert selected.year.eq(2040).all()
    national[kind]=selected.set_index('metric').value_pj.to_dict()
    source_files.append(path)
    display(selected[['metric','value_pj','source_id','source_location']])
g,d=national['g'],national['d']
display(pd.DataFrame([{'Datei':str(XLSX),'Flächenspalte':AREA_COLUMN,'Typen':len(data),
 'positive Typen':int(area.gt(0).sum()),'BGF [Mio. m²]':area.sum()/1e6,'SHA256':sha256(XLSX)}]))


,metric,value_pj,source_id,source_location
18,fossil,1,TRANSITION,"Tabelle 16, S. 63"
19,hydro,173,TRANSITION,"Tabelle 16, S. 63"
20,biomass,21,TRANSITION,"Tabelle 16, S. 63"
21,geothermal,4,TRANSITION,"Tabelle 16, S. 63"
22,pv,149,TRANSITION,"Tabelle 16, S. 63"
23,wind,105,TRANSITION,"Tabelle 16, S. 63"
24,hydrogen_electricity,4,TRANSITION,"Tabelle 16, S. 63"
25,generation_published_total,457,TRANSITION,"Tabelle 16, S. 63"
26,net_imports,-7,TRANSITION,"Tabelle 16, S. 63"


,metric,value_pj,source_id,source_location
16,verkehr,90,TRANSITION,"Tabelle 15, S. 60"
17,gebaeude,108,TRANSITION,"Tabelle 15, S. 60"
18,landwirtschaft,4,TRANSITION,"Tabelle 15, S. 60"
19,industrie,140,TRANSITION,"Tabelle 15, S. 60"
20,verbrauch_sektor_energie,34,TRANSITION,"Tabelle 15, S. 60"
21,transportverluste,19,TRANSITION,"Tabelle 15, S. 60"
22,umwandlungseinsatz,57,TRANSITION,"Tabelle 15, S. 60"
23,electricity_demand_published_total,450,TRANSITION,"Tabelle 15, S. 60"


,Datei,Flächenspalte,Typen,positive Typen,BGF [Mio. m²],SHA256
0,C:\Users\schneids\code\peq-higrube\peq-higrube...,Nationale BGF 2040,900,603,1126.578658,b032c1bd7148b0bdac8541196a1eaa7fc58f538f87912e...


## Dichtezuordnung und Erwartungswert

Ein einziges Ziel erhält 100 %. Bei zwei Zielen müssen beide Anteile vorhanden sein
und zusammen 100 % ergeben. Identische zusammengefasste Statistiknutzungen werden
nicht doppelt gezählt. Die GFZ-Verteilung 2021 bleibt ein vorläufiges räumliches Muster.

$$E[c_D\mid k]=\sum_j s_{kj}\,\left[-\min\left(\frac{f_{RES}}{GFZ_j+dx}-f_{ED},c_{max}\right)\right].$$

$s_{kj}$ ist der bedingte BGF-Anteil im GFZ-Bin, $k$ die Geometrieklasse.
Die historische Kurve (Schneider et al. 2023, Tabelle 5/Gleichung 7) wird wie im
wissenschaftlichen Anhang vorläufig als EE-Strom interpretiert. Sie ist ein
**virtueller Bewertungswert**, keine Stromerzeugung und keine bestätigte neue Kalibrierung.


In [3]:
use_map={'Gebäude für Gemeinschaften':'Gemeinschaftsgebäude','Hotels und ähnliche Gebäude':'Hotels o.Ä.',
 'Bürogebäude':'Büronutzung','Groß- und Einzelhandelsgebäude':'Groß- und Einzelhandelsgebäude',
 'Gebäude des Verkehrs- und Nachrichtenwesens':'Verkehrs- und Nachrichtenwesen','Industrie- und Lagergebäude':'Industrie und Lagerung',
 'Gebäude für Kultur- und Freizeitzwecke sowie des Bildungs- und Gesundheitswesens':'Kultur, Freizeit, Bildung und Gesundheit',
 'Wohngebäude mit einer Wohnung':'Ein- und Zweifamilienhäuser','Wohngebäude mit zwei Wohnungen':'Ein- und Zweifamilienhäuser',
 'Wohngebäude mit drei und mehr Wohnungen':'Mehrfamilienhaus'}
mapping['GWR_Nutzung']=mapping.Nutzung.map(use_map); assert mapping.GWR_Nutzung.notna().all()
mapping['GK']=mapping.Geschoss.str.replace(' Geschoße','',regex=False).str.replace(' Geschoß','',regex=False).replace({'6 bis 10':'6-10','11 und mehr':'11+'})
mapping['Einzelziel_ergänzt']=mapping.Anteil1.isna() & mapping.Archetype1.notna() & mapping.Archetype2.isna()
if True:
    mapping.loc[mapping['Einzelziel_ergänzt'],'Anteil1']=1.0
# Doppelte Statistiknutzungen nur zusammenführen, wenn ihre Regeln identisch sind.
for key,group in mapping.groupby(['GWR_Nutzung','GK']):
    assert len(group[['Anteil1','Archetype1','Anteil2','Archetype2']].drop_duplicates())==1,key
unique=mapping.drop_duplicates(['GWR_Nutzung','GK'])
long=[]
for _,row in unique.iterrows():
    for n in [1,2]:
        if pd.notna(row[f'Archetype{n}']):
            long.append({'GWR_Nutzung':row.GWR_Nutzung,'GK':row.GK,'Geometrie':row[f'Archetype{n}'],'Anteil':row[f'Anteil{n}']})
links=pd.DataFrame(long)
assert links.Anteil.notna().all(), 'GFZ-Anteile offen; Einzelzielregel oder Eingaben prüfen'
assert links.Anteil.between(0,1).all()
np.testing.assert_allclose(links.groupby(['GWR_Nutzung','GK']).Anteil.sum(),1)
gfz=pd.read_csv(ROOT/'Berechnungen/gfz_at/outputs/recalculated/gfz_distribution_by_province_use_storeys.csv',dtype={'GK':str})
joined=gfz.merge(links,on=['GWR_Nutzung','GK'],how='left',indicator=True)
assert joined['_merge'].eq('both').all()
joined['zugeordnete_BGF']=joined.BGF_m2*joined.Anteil
np.testing.assert_allclose(joined.zugeordnete_BGF.sum(),gfz.BGF_m2.sum(),rtol=1e-12)
bins=joined.groupby(['Geometrie','GFZ'],as_index=False).zugeordnete_BGF.sum()
bins['s']=bins.zugeordnete_BGF/bins.groupby('Geometrie').zugeordnete_BGF.transform('sum')
assert set(data.loc[data.BGF.gt(0),'Geometrie']).issubset(set(bins.Geometrie)), 'GFZ-Zuordnung fehlt'
source_files.append(ROOT/'Berechnungen/gfz_at/outputs/recalculated/gfz_distribution_by_province_use_storeys.csv')
display(links)


,GWR_Nutzung,GK,Geometrie,Anteil
0,Gemeinschaftsgebäude,1,NiWo_1G,1.0
1,Gemeinschaftsgebäude,2,NiWo_2G,1.0
2,Gemeinschaftsgebäude,3,NiWo_3G,1.0
3,Gemeinschaftsgebäude,4,NiWo_3G,1.0
4,Gemeinschaftsgebäude,5,NiWo_3G,1.0
...,...,...,...,...
62,Mehrfamilienhaus,3,RH_2G,0.5
63,Mehrfamilienhaus,4,MFH_4G,1.0
64,Mehrfamilienhaus,5,GWB_5G,1.0
65,Mehrfamilienhaus,6-10,GWB_7G,1.0


## Rechenmodell und Bilanzkontrollen

$$D_B=\sum_k A_{BGF,k}\,r\,e_{el,k}\,(1-u),\quad
P_B=\sum_k A_{BGF,k}\,r\,e_{PV,k}\,v,\quad B_B=P_B-D_B.$$

$r$ = NGF/BGF, $u$ = zusätzliche Strombedarfssenkung, $v$ = PV-Skalierung.
Beide Eingriffe sind Szenariotests, keine neue Gebäude- oder Eigenverbrauchssimulation.
Sie verändern keine thermischen oder primärenergetischen Kennwerte.

Zentrale budgetfähige Erzeugung bedient zunächst Industrie, Landwirtschaft,
Energiebranche, Umwandlung, Netzverluste und Exporte sowie ÖPNV und übrigen Verkehr.
Das verbleibende MIV-Budget ist auf 0 bis MIV-Bedarf begrenzt. Ein negativer Rest
bleibt als Prioritätskonflikt sichtbar. Freiflächen-PV wird **nicht automatisch** in
den zentralen Pool aufgenommen; eine solche Regel wäre eine zusätzliche normative Entscheidung.

$$T_B=P_{PV,TD}(1-f)-D_{B,TD},\qquad
T_{B+MIV}=T_B+G_{MIV}-D_{MIV}.$$

Der Vergleich verwendet auf beiden Seiten dieselbe MIV-Zuordnung. Daher ist
$\Delta=B_B-T_B$ unabhängig vom MIV-Anteil. Dichtewerte werden separat gezeigt,
nicht zur Schließung dieser physischen Differenz addiert. PEQ-Anteile und alternative
Typenkennwerte müssen aus den Eingabedaten kommen; das Dashboard erfindet keine Typenumwandlung.


In [4]:
generation_keys=['hydro','wind','biomass','geothermal','hydrogen_electricity','pv','fossil']
demand_keys=['verkehr','gebaeude','industrie','landwirtschaft','verbrauch_sektor_energie','transportverluste','umwandlungseinsatz']
residual=sum(g[k] for k in generation_keys)+g['net_imports']-sum(d[k] for k in demand_keys)
# Aggregation in PJ bei r=1; der editierbare Flächenfaktor wird erst im Dashboard angewandt.
active=data.loc[data.BGF.gt(0)].copy()
active['D']=active.BGF*active.demand*3.6e-9
active['P']=active.BGF*active.pv*3.6e-9
geometry=active.groupby('Geometrie').agg(BGF=('BGF','sum'),D=('D','sum'),P=('P','sum'))
def calculate(p):
    assert all(np.isfinite(v) for v in p.values())
    assert all(0<=p[k]<=1 for k in ['miv','other_transport','free_pv','hydrogen','saving'])
    assert p['miv']+p['other_transport']<=1
    assert p['ngf_bgf']>0 and p['pv_scale']>=0 and p['density_dx']>0
    assert all(p[k]>=0 for k in ['density_f_res','density_f_ed','density_cutoff'])
    central=sum(g[k] for k in ['hydro','wind','biomass','geothermal'])+g['hydrogen_electricity']*p['hydrogen']
    fixed=sum(d[k] for k in ['industrie','landwirtschaft','verbrauch_sektor_energie','umwandlungseinsatz','transportverluste'])+max(-g['net_imports'],0)
    raw=central-fixed-d['verkehr']*(1-p['miv'])
    miv=d['verkehr']*p['miv']; budget=min(max(raw,0),miv)
    target=g['pv']*(1-p['free_pv'])-d['gebaeude']
    demand=geometry.D.sum()*p['ngf_bgf']*(1-p['saving'])
    pv=geometry.P.sum()*p['ngf_bgf']*p['pv_scale']
    unassigned=sum(g[k] for k in generation_keys)-g['pv']-central
    closure=target+budget-miv+raw-budget+g['pv']*p['free_pv']+unassigned+max(g['net_imports'],0)
    assert np.isclose(closure,residual)
    return dict(demand=demand,pv=pv,balance=pv-demand,target=target,budget=budget,
                extended=pv-demand+budget-miv,target_extended=target+budget-miv,
                gap=pv-demand-target,raw=raw,closure=closure)
reference=calculate(DEFAULT)
for m,f in [(0,0),(1,1),(.6,.2)]:
    result=calculate(dict(DEFAULT,miv=m,free_pv=f))
    assert np.isclose(result['extended']-result['target_extended'],result['gap'])
display(pd.DataFrame([reference],index=['Referenz; PJ_el/a']).T)
payload=dict(defaults=DEFAULT,reference=reference,g=g,d=d,residual=residual,
             geometry=geometry.reset_index().to_dict('records'),
             bins=bins[['Geometrie','GFZ','s']].to_dict('records'),
             provenance={str(p.relative_to(ROOT)) if p.is_relative_to(ROOT) else str(p):sha256(p) for p in source_files},
             scenario=SCENARIO,area_column=AREA_COLUMN,xlsx=str(XLSX))


,Referenz; PJ_el/a
demand,137.493260
pv,78.245462
balance,-59.247798
target,41.000000
budget,24.000000
extended,-107.247798
target_extended,-7.000000
gap,-100.247798
raw,24.000000
closure,-2.000000


## Dashboard: verändern, vergleichen, sichern

Die Startwerte entsprechen dem dokumentierten Rechenbeispiel, **keiner normativen Freigabe**.
Gebäude-PV/ÖPNV werden jeweils als Restanteile berechnet. „Übriger Verkehr“ bleibt
eigenständig. Grün/Blau kennzeichnen Erzeugung und Bedarf, Orange den Zielwert,
Violett virtuelle Kontextwerte. Alle Diagramme verwenden die aktuelle Parametrierung.

Die angezeigte zusätzliche PV-Lücke ist ein algebraischer Rest bei konstantem Bedarf,
kein Standort-, Flächen- oder Machbarkeitsnachweis. Die Dichtekurve darf verändert werden,
aber wird nicht automatisch optimiert. Die finale Publikation verwendet weiterhin nur
die bewusst übernommenen wissenschaftlichen Modellstände.


Die Kennzahlen stehen in drei Spalten: **Bedarf, Deckung und Saldo**. Gebäudebetrieb und zugeordnete Mobilität addieren sich spaltenweise zur Bottom-Up-Summe. Darunter steht der Top-Down-Vergleich mit identischer MIV-Zuordnung. Hover-Erklärungen an Reglern und Kacheln nennen Bezugsgrößen und Rechenregeln. TD bezeichnet das nationale Top-Down-Szenario.

In [5]:
# Technische HTML-Darstellung; die wissenschaftliche Referenzrechnung steht oben.
template=(BASE/'dashboard_template.html').read_text(encoding='utf-8')
encoded=json.dumps(payload,ensure_ascii=False,allow_nan=False).replace('<',r'\u003c')
display(HTML(template.replace('__PAYLOAD__',encoded)))
